# 02c — Medication features

Identify qualifying recent antibiotic exposure. One row is retained per hospital spell.

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.io import load_pkg_yaml

schema = load_pkg_yaml("config/icare/schema.yaml")
DATA_DIR = Path(schema["data_root"])
KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]

def keep_first_spell(df: pd.DataFrame, name: str = "episodes") -> pd.DataFrame:
    """Keep the first source row for each SUBJECT/SPELL_IDENTIFIER pair."""
    missing = [c for c in KEYS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing key columns: {missing}")
    result = df.copy()
    before = len(result)
    result = result.drop_duplicates(KEYS, keep="first").reset_index(drop=True)
    print(f"{name}: {before:,} rows -> {len(result):,} unique spells")
    return result

print(f"Using data from: {DATA_DIR}")


Using data from: /app/notebooks/data/synthetic/10000p


In [4]:
from icare_risk.clinphen.utils import get_qualifying_last_abx

df_episodes = pd.read_parquet(DATA_DIR / "icare_episodes_anon.parquet")
df_episodes = keep_first_spell(df_episodes)
df_episodes["ADMISSION_DATE"] = pd.to_datetime(df_episodes["ADMISSION_DATE"])
df_episodes["DISCHARGE_DATE"] = pd.to_datetime(df_episodes["DISCHARGE_DATE"])

df_administration = pd.read_csv(DATA_DIR / "icare_pharmacy_administration_anon.csv")


episodes: 10,000 rows -> 10,000 unique spells


In [5]:
target_abx = [
    "co-amoxiclav", "ciprofloxacin", "ceftriaxone",
    "ertapenem", "meropenem",
]

df_medications = get_qualifying_last_abx(
    df_episodes=df_episodes,
    df_admin=df_administration,
    target_abx=target_abx,
    subject_col="SUBJECT",
    min_duration_hours=24,
    verbose=True,
)
df_medications = keep_first_spell(df_medications, "medication features")
display(df_medications.head())


--- Qualifying Last Antibiotic Summary (> 24h) ---
  Qualifying ABX Events :       0 / 10,000 (0.0%)

  No qualifying antibiotics identified.
--------------------------------------------------
medication features: 10,000 rows -> 10,000 unique spells


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,qualifying_last_abx
0,11922,3899275,2176201,7630712,2020-01-01,03:55:00,45,2020-01-05,300,general medicine,3,NaN
1,17758,2504729,6044210,7507696,2020-01-01,08:11:00,71,2020-01-14,140,oral surgery,8,NaN
2,13207,2309018,6263689,1862979,2020-01-01,08:34:00,75,2020-01-25,320,cardiology,9,NaN
3,16388,4790491,1590221,3991653,2020-01-01,11:27:00,48,2020-01-05,140,oral surgery,none,NaN
4,12838,6160669,7583605,2532114,2020-01-01,14:22:00,83,2020-01-02,320,cardiology,8,NaN


In [6]:
output_path = DATA_DIR / "02d_derived_features_medications.parquet"
df_medications.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /app/notebooks/data/synthetic/10000p/02d_derived_features_medications.parquet
